# AI-Powered Parcel Damage Inspection

**Presentation demo — four-stage, human-in-the-loop computer vision**

This notebook runs the project's current parcel-level inference pipeline:

```text
phone image -> parcel detector -> parcel crop -> open/closed classifier
    open -> REVIEW
    closed -> damaged/intact classifier -> optional damage localization
```

It performs inference only: no training, threshold tuning, label changes, or final-test access.
A GPU is recommended for a smooth presentation, but CPU fallback is supported.

## 1. Runtime check and dependency

In Colab, choose **Runtime → Change runtime type → T4 GPU** when one is available.
Colab GPU availability varies, so the code reports the actual device and safely falls back to CPU.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.129"
])
print("Dependency installed.")

In [ ]:
import platform
import torch
import torchvision

print(f"Python:      {platform.python_version()}")
print(f"PyTorch:     {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA ready:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
else:
    print("GPU is unavailable; the demo will run on CPU.")

## 2. Get the slim demo project

The default mode clones the curated GitHub repository. If the repository is private or has not
been pushed yet, choose `upload_bundle` and upload `parcel_damage_demo_bundle.zip` when prompted.
The bundle is about 40 MB and contains only the inference code, frozen weights, and development
examples—not the 1+ GB training datasets.

In [ ]:
#@title Project source
SOURCE = "github" #@param ["github", "upload_bundle"]
REPOSITORY_URL = "https://github.com/yacine1605/project_parcel.git" #@param {type:"string"}
REPOSITORY_BRANCH = "main" #@param {type:"string"}

import shutil
import subprocess
import zipfile
from pathlib import Path

WORKSPACE = Path("/content") if Path("/content").is_dir() else Path.cwd()

def safe_extract(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            candidate = (destination / member.filename).resolve()
            if destination != candidate and destination not in candidate.parents:
                raise ValueError(f"Unsafe archive path: {member.filename}")
        archive.extractall(destination)

if SOURCE == "github":
    PROJECT_ROOT = WORKSPACE / "project_parcel"
    if not PROJECT_ROOT.is_dir():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH,
            REPOSITORY_URL, str(PROJECT_ROOT),
        ])
    else:
        print(f"Reusing existing checkout: {PROJECT_ROOT}")
else:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Bundle upload mode must run in Google Colab.") from error
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one parcel_damage_demo_bundle.zip file.")
    uploaded_bundle = WORKSPACE / zip_names[0]
    uploaded_bundle.write_bytes(uploaded[zip_names[0]])
    safe_extract(uploaded_bundle, WORKSPACE)
    PROJECT_ROOT = WORKSPACE / "parcel_damage_demo"

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root was not created: {PROJECT_ROOT}")
print(f"Project root: {PROJECT_ROOT}")

## 3. Verify model artifacts

SHA-256 checks protect the presentation from accidentally loading a different checkpoint.
All four checkpoint hashes are verified before any model is loaded.

In [ ]:
import hashlib
import json

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest_path = PROJECT_ROOT / "colab" / "asset_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for model_record in manifest["models"]:
    model_path = PROJECT_ROOT / model_record["file"]
    if not model_path.is_file():
        raise FileNotFoundError(f"Missing model: {model_path}")
    observed = file_sha256(model_path)
    if observed != model_record["sha256"]:
        raise RuntimeError(f"Hash mismatch: {model_record['name']}")
    print(f"PASS  {model_record['role']}: {model_record['name']} ({model_record['version']})")

## 4. Load the four inference components

The generic detector finds parcels. A crop classifier identifies open boxes. Closed parcels then
receive damaged/intact classification, and damage YOLO localizes defects only when requested.

In [ ]:
import os
import sys

os.environ["YOLO_CONFIG_DIR"] = str(PROJECT_ROOT / "Ultralytics")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from prototype.inspection_pipeline import (
    inspect_parcel_image,
    load_prototype_models,
    open_supported_image,
)

loaded_models = load_prototype_models()
print("All four inference components loaded successfully.")

## 5. Run the prepared presentation examples

These are labeled **validation/development examples**, not fresh evidence of model quality. They
demonstrate damage localization, open-box gating, a first-stage detector miss, and a state-classifier
domain-shift error. Failures are retained because understanding them is part of the project.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def inspect_and_display(image_path: Path, title: str | None = None):
    image = open_supported_image(image_path)
    evidence, annotated = inspect_parcel_image(image, loaded_models)

    summary = {
        "image": title or image_path.name,
        "status": evidence["status"],
        "parcels": len(evidence["parcels"]),
        "parcel_states": ", ".join(p["parcel_state"] for p in evidence["parcels"]) or "none",
        "classifier": evidence["classifier_label"] if evidence["classifier_executed"] else "not run",
        "damaged_probability": (
            evidence["classifier_probability"] if evidence["classifier_executed"] else None
        ),
        "yolo_detections": evidence["yolo_detection_count"],
        "damage_types": ", ".join(evidence["yolo_damage_types"]) or "none",
        "decision": evidence["prototype_decision"],
        "total_ms": evidence["timing_ms"]["total_pipeline"],
    }
    return evidence, annotated, summary

prepared_results = []
figure, axes = plt.subplots(2, 2, figsize=(13, 12))
for axis, sample in zip(axes.flat, manifest["samples"]):
    evidence, annotated, summary = inspect_and_display(
        PROJECT_ROOT / sample["file"], sample["title"]
    )
    prepared_results.append(summary)
    axis.imshow(annotated)
    axis.set_title(
        f"{sample['title']} — {evidence['prototype_decision']}\n"
        f"ground truth: {sample['ground_truth']}"
    )
    axis.axis("off")
plt.tight_layout()
plt.show()

prepared_frame = pd.DataFrame(prepared_results)
prepared_frame["damaged_probability"] = prepared_frame["damaged_probability"].map(
    lambda value: "not run" if pd.isna(value) else f"{value:.1%}"
)
prepared_frame["total_ms"] = prepared_frame["total_ms"].map(lambda value: f"{value:.1f}")
display(prepared_frame)

### Demo-image attribution

The prepared examples are derivative validation images from the following Roboflow Universe datasets,
both provided under **CC BY 4.0**:

- [Damaged Box Detection](https://universe.roboflow.com/project-33xgh/damaged-box-detection)
- [My First Project / damaged-package-detection](https://universe.roboflow.com/damaged-package-detection/my-first-project-5g94h)

See `colab/ATTRIBUTION.md` in the project for details.

## 6. Inspect your own parcel image

Run the next cell, upload one JPG/PNG/WebP image, and the notebook will show the annotated evidence.
Images stay in the temporary Colab runtime unless you explicitly download the result.

In [ ]:
from google.colab import files

uploaded_images = files.upload()
if len(uploaded_images) != 1:
    raise ValueError("Upload exactly one parcel image.")

uploaded_name, uploaded_bytes = next(iter(uploaded_images.items()))
uploaded_path = WORKSPACE / Path(uploaded_name).name
uploaded_path.write_bytes(uploaded_bytes)

custom_evidence, custom_annotated, custom_summary = inspect_and_display(uploaded_path)
display(pd.DataFrame([custom_summary]))
display(custom_annotated)

## 7. Download the evidence (optional)

The JSON keeps every model output and timing value. The annotated image provides visual evidence
for an operator. The notebook does not claim a severity score or automatically reject a parcel.

In [ ]:
from google.colab import files

evidence_path = WORKSPACE / "parcel_inspection_evidence.json"
annotated_path = WORKSPACE / "parcel_inspection_annotated.png"
evidence_path.write_text(json.dumps(custom_evidence, indent=2), encoding="utf-8")
custom_annotated.save(annotated_path)

files.download(str(evidence_path))
files.download(str(annotated_path))

## Results to present honestly

- **MobileNetV3-Large final test (592 images):** 89.29% damaged recall, 89.06% precision,
  89.17% F1, 93.35% ROC-AUC, and 78.50% specificity.
- **YOLO11n EXP-002 validation (444 images / 681 instances):** 0.584 precision, 0.561 recall,
  0.588 mAP50, and 0.296 mAP50–95.
- **Parcel-state classifier validation (66 crops):** macro-F1 0.8019; this small result is preliminary.
- Each metric belongs to its component dataset. The combined four-stage policy has not been validated
  as a production system.
- Open parcels, missing localization, model disagreement, and detected damage go to **REVIEW**. The
  prototype never issues **REJECT** or invents physical severity.
- The project is a research prototype, not a validated warehouse safety system.

The strongest presentation story is not “the model is perfect.” It is that the system keeps raw
evidence, exposes known failure modes, and uses a conservative human-in-the-loop decision.